# Imports

In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 11 16:05:15 2023

@author: andrej
"""
import sys,os
sys.path.append(r'Z:/EnergyTrading/Python/')

import pandas as pd
from Database.TPData import TPData
from Database.DB_reader import Database
from datetime import date, timedelta

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\user\Downloads\instantclient_21_11")
except:
    pass

# Email utilities

In [2]:
from Utilities.email_sending import send_plain_email, send_html_email

EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = "zubal_andrej@energytrading.sk" # can also be a list of recipients

# Defining date range

In [3]:
def date_range(start_dt, end_dt):

    # difference between current and previous date
    delta = timedelta(days=1)

    # store the dates between two dates in a list
    dates = []

    while start_dt <= end_dt:
        # add current date to list by converting  it to iso format
        dates.append(start_dt.isoformat())
        # increment start date by timedelta
        start_dt += delta

    return dates

#date_range=date_range(date(2019, 4, 4), date(2023, 10, 11))
date_range=date_range(date.today()+timedelta(-1), date.today()+timedelta(-1))
print(date_range)

['2025-04-10']


# Loading data and sending emails

In [4]:
local_db_config_path=r'Z:\EnergyTrading\configDB.json'

try:
    for date in date_range:
        
        conn = Database('OracleSQL',path_name=local_db_config_path)

        query=f"""select * from  rove_od.trayport_vw_trades 
        WHERE 1=1
        and to_date(datetime)>=to_date('{date}', 'YYYY-MM-DD')
        and to_date(datetime)<=to_date('{date}', 'YYYY-MM-DD')""" # where rownum <= 100"""
        #query="""select * from  public.trayport_orders limit 100"""

        df1=conn.execute(query)

        conn = Database(path_name=local_db_config_path)
        conn._connect()
        df1.to_sql('trayport_vw_trades', conn.engine, schema='public', if_exists='append', index=False)

        print('\n')
        print(f'{date} number of records: {df1.shape[0]}')
        
        conn.execute_general_query("CALL public.refresh_mv_trade_data();")
        conn.execute_general_query("CALL public.refresh_mv_mistrade_data();")
        
        #also updating timescaleDB table
        conn = Database('timescaledb',path_name=local_db_config_path)
        conn._connect()
        df1.to_sql('trades', conn.engine, schema='public', if_exists='append', index=False)

        #email sending in case of success, also sending number of records uploaded
        send_plain_email(
          RECIPIENT, 
        "SUCCESS: trade_data_daily_update_remote_comp job", f'{date} number of records: {df1.shape[0]}',
        email_password=EMAIL_PASSWORD
    )
        print('\n')

        conn._disconnect()

except:
    send_plain_email(
          RECIPIENT, 
        "FAIL: trade_data_daily_update_remote_comp", f'The upload of data failed for this run, please check what is the issue.',
        email_password=EMAIL_PASSWORD
    )

Connected to the database oracle
Disconnected from the database oracle
Connected to the database timescaledb


Disconnected from the database timescaledb
